# SNN full study on Kaggle

Before running:

- `git push` your repo first (Kaggle clones from GitHub).
- Settings: GPU (T4 x2 / P100), Internet On, Persistence = Variables and Files.
- One mechanism per session (~9h limit); persist `results/` as a Dataset (last cell).
- Edit `REPO_URL` in the next cell.

In [ ]:
# 1. Clone + install
import os, subprocess, sys

REPO_URL = "https://github.com/thisisrick25/snn.git"  # EDIT ME (private repo: use a token URL)
REPO_DIR = "/kaggle/working/snn"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
os.environ["PYTHONPATH"] = REPO_DIR

# torch/torchvision/numpy ship with Kaggle images; do NOT pin +cu132 (code is device-agnostic via device: auto)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "snntorch", "pyyaml"], check=True)

import torch
print("cwd:", os.getcwd())
print("cuda available:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

In [ ]:
# 2. (Optional) Restore prior results from a Kaggle Dataset input, so --resume can skip finished work.
# Add your saved dataset via 'Add Input' (right panel); it mounts read-only under /kaggle/input/<slug>/.
import glob, os, shutil

dest = "results/full_study/raw"
os.makedirs(dest, exist_ok=True)
restored = 0
raw_dirs = (glob.glob("/kaggle/input/*/results/full_study/raw")
            + glob.glob("/kaggle/input/*/out/full_study/raw")
            + glob.glob("/kaggle/input/**/full_study/raw", recursive=True)
            + glob.glob("/kaggle/input/*/raw"))
for raw_dir in sorted(set(raw_dirs)):
    for f in glob.glob(os.path.join(raw_dir, "*.json")):
        shutil.copy(f, dest)
        restored += 1
print(f"restored {restored} raw JSON(s) into {dest}")

In [ ]:
# 3. Run the whole 12-cell matrix + final analysis via the skip-aware wrapper.
# It walks (mnist/mlp, cifar10/conv_snn) x (kwta_window, activity_reg, threshold)
# core cells + the 3 confound controls at threshold, then runs run_confirmatory.
# --resume is ON: completed cells and finished (seed, condition) pairs are skipped,
# so just re-run this cell each session and it continues where it left off.
!bash run_full_study.sh

In [ ]:
# 3b. (Alternative) Run ONE cell of the matrix instead of the whole wrapper.
# DATASET = "mnist"; ARCH = "mlp"; SPARSITY = "kwta_window"; CONTROL = "none"
# cmd = ["python", "-m", "src.scripts.run_pilot", "--config", "configs/full_study.yaml",
#        "--dataset", DATASET, "--arch", ARCH, "--sparsity-mode", SPARSITY]
# if CONTROL != "none": cmd += ["--control", CONTROL]
# import subprocess; subprocess.run(cmd, check=True)

In [ ]:
# 4. Progress check: how many raw JSONs per cell (expected: kwta 36, activity_reg 36, threshold 81, each control 9).
import glob
from collections import Counter

counts = Counter()
for f in glob.glob("results/full_study/raw/*.json"):
    parts = os.path.basename(f).split("_")
    if len(parts) >= 5:
        counts["_".join(parts[1:5])] += 1  # dataset_arch_mechanism_control
for k in sorted(counts):
    print(f"{counts[k]:>3}  {k}")

In [ ]:
# 5. Final analysis. The wrapper already runs this once all cells are done;
# use this to re-run/inspect it on its own.
import subprocess
subprocess.run(["python", "-m", "src.scripts.run_confirmatory", "--config", "configs/full_study.yaml"], check=True)
print(open("results/full_study/metrics/confirmatory.json").read())

In [ ]:
# 6. Save results so the next session can resume.
# Copies results/full_study into /kaggle/working/out (kept when you 'Save Version').
# For cross-session resume, also create/version a Kaggle Dataset from this folder, then
# 'Add Input' it next session so cell 2 can restore it.
import shutil, os
out = "/kaggle/working/out/full_study"
if os.path.isdir(out):
    shutil.rmtree(out)
shutil.copytree("results/full_study", out)
print("saved to", out)

## Notes

- Quota: ~30 GPU-h/week is enough for the full matrix, but one session caps at ~9-12h, so expect 2-3 sessions (CIFAR-Conv-SNN is the slow part). Re-run cell 3 each session; the wrapper resumes.
- Persistence = Files keeps `/kaggle/working` across sessions of the *same* notebook, so `results/full_study/raw` survives and `--resume` continues. It can be lost on reset/fork/container recycle, so also save a Dataset (cell 6) as a backup and re-add + re-run cell 2 next session.
- Let Kaggle generate activity_reg fresh; don't import old local `*_activity_reg_*.json`.
- To redo a cell: delete its raw JSONs and re-run, or use cell 3b with `--no-resume`.